In [ ]:
import torch
import pandas as pd
import numpy as np
from ucimlrepo import fetch_ucirepo
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
import time
from gdv_utils import calculate_gdv 

# 1. SETUP DEVICE PER LA GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo per analisi GDV: {device}\n")

# 2. MAPPING DEI DATASET COMPLETO (Nome -> UCI ID)
UCI_DATASETS = {
    "Pen-Based Recognition of Handwritten Digits": 81,
    "Breast Cancer Coimbra": 451,
    "Page Blocks Classification": 78,
    "Maternal Health Risk": 863,
    "Molecular Biology (Splice-junction Gene Sequences)": 69,
    "Spambase": 94,
    "Steel Plates Faults": 198,
    "Bank Marketing": 222,
    "Blood Transfusion Service Center": 176,
    "Raisin": 850,
    "Website Phishing": 379,
    "Letter Recognition": 59,
    "Taiwanese Bankruptcy Prediction": 572,
    "Waveform Database Generator (Version 1)": 107,
    "Statlog (Image Segmentation)": 147,
    "Haberman’s Survival": 43,
    "Vertebral Column": 212,
    "Statlog (German Credit Data)": 144,
    "Optical Recognition of Handwritten Digits": 80,
    "Breast Cancer": 14,
    "Drug Consumption (Quantified)": 373,
    "Mammographic Mass": 161,
    "Yeast": 110,
    "Credit Approval": 27,
    "Contraceptive Method Choice": 30,
    "Hepatitis C Virus (HCV) for Egyptian patients": 503,
    "Japanese Credit Screening": 28,
    "Chess (King-Rook vs. King-Pawn)": 22,
    "Student Performance on an Entrance Examination": 582,
    "Predict Students’ Dropout and Academic Success": 697,
    "Heart Disease": 45,
    "SPECT Heart": 95,
    "Room Occupancy Estimation": 864,
    "Differentiated Thyroid Cancer Recurrence": 915,
    "ISOLET": 54,
    "Statlog (Vehicle Silhouettes)": 149,
    "Musk (Version 2)": 75,
    "National Poll on Healthy Aging (NPHA)": 936,
    "Breast Cancer Wisconsin (Diagnostic)": 17,
    "Hayes-Roth": 44,
    "Congressional Voting Records": 105,
    "Cardiotocography": 193,
    "Cirrhosis Patient Survival Prediction": 878,
    "Autism Screening Adult": 426,
    "SPECTF Heart": 96,
    "Statlog (Heart)": 145,
    "Image Segmentation": 50,
    "ILPD (Indian Liver Patient Dataset)": 225,
    "NHANES 2013-2014 Age Prediction Subset": 887,
    "Statlog (Australian Credit Approval)": 143,
    "Ionosphere": 52,
    "Polish Companies Bankruptcy": 365
}

def clean_and_prepare_data(X, y, max_samples=15000):
    """
    Pipeline di pulizia universale per i dataset UCI.
    Include sanificazione avanzata, inferenza dei tipi, 
    rimozione ID/Leakage e limite campioni.
    """
    # 1. Standardizzazione formati in Pandas DataFrame
    if not isinstance(X, pd.DataFrame):
        X = pd.DataFrame(X)
    if not isinstance(y, (pd.DataFrame, pd.Series)):
        y = pd.Series(y.ravel() if hasattr(y, 'ravel') else y)

    # Gestione Multi-Target (prendiamo solo il primo target per sicurezza)
    if isinstance(y, pd.DataFrame) and y.shape[1] > 1:
        y = y.iloc[:, 0]

    # =================================================================
    # FIX: ANTI-ESPLOSIONE CARTESIANA
    # Resettiamo gli indici per garantire che siano univoci (0, 1, 2... N)
    # Questo previene la moltiplicazione delle righe durante i successivi .loc
    X = X.reset_index(drop=True)
    y = y.reset_index(drop=True)
    # =================================================================

    # 2. Rimozione dei NaN sui target (Ora è sicuro al 100%)
    valid_idx = y.dropna().index
    X = X.loc[valid_idx].copy()
    y = y.loc[valid_idx].copy()

    # Limite di 15000 campioni per stabilità statistica e RAM
    if len(X) > max_samples:
        sampled_idx = X.sample(n=max_samples, random_state=42).index
        X = X.loc[sampled_idx]
        y = y.loc[sampled_idx]
    
    # =================================================================
    # 3. --- SANIFICAZIONE E INFERENZA DEI TIPI AVANZATA (BISTURI) ---
    X = X.replace(['?', 'NA', 'null', 'None', '', ' '], np.nan)

    for col in X.columns:
        if not pd.api.types.is_numeric_dtype(X[col]):
            valori_presenti = X[col].notna().sum()
            if valori_presenti > 0:
                # Tentativo di conversione con fix per i decimali europei
                tentativo = pd.to_numeric(X[col].astype(str).str.replace(',', '.'), errors='coerce')
                numeri_validi = tentativo.notna().sum()
                # Se il 90% dei dati presenti è numerico, salva la colonna come numero
                if (numeri_validi / valori_presenti) > 0.90:
                    X[col] = tentativo

    # 4. --- BLACKLIST DATA LEAKAGE E RIMOZIONE ID ---
    X = X.drop(columns=['result'], errors='ignore')

    colonne_rimosse = []
    for col in list(X.columns):
        
        # Regola A: ID espliciti (tutti valori unici)
        if X[col].nunique() == len(X):
            
            # 1. Se è Testo (es. "Nome_Paziente", "Hash"), lo droppiamo subito
            if not pd.api.types.is_numeric_dtype(X[col]):
                colonne_rimosse.append(col)
                X = X.drop(columns=[col])
                
            # 2. Se è un Intero (int), potrebbe essere una misura (Pixel) o un ID (1,2,3...)
            elif pd.api.types.is_integer_dtype(X[col]):
                # Gli ID veri sono quasi sempre sequenziali. Le misure fisiche no.
                if X[col].is_monotonic_increasing or X[col].is_monotonic_decreasing:
                    colonne_rimosse.append(col)
                    X = X.drop(columns=[col])
                    
        # Regola B: Testo con alta cardinalità (>50% valori unici)
        elif not pd.api.types.is_numeric_dtype(X[col]):
            if X[col].nunique() > (len(X) * 0.5):
                colonne_rimosse.append(col)
                X = X.drop(columns=[col])
    
    # 5. Encoding categoriali 
    # (Ora è sicuro usare get_dummies perché le colonne sporche/ID sono state rimosse)
    X = pd.get_dummies(X, dtype=float)
    # 6. Imputazione e Scaling
    imputer = SimpleImputer(strategy='mean')
    X_imputed = imputer.fit_transform(X)

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_imputed)

    # 7. Encoding Labels
    encoder = LabelEncoder()
    y_encoded = encoder.fit_transform(y.values.ravel())

    # 8. Spostamento su device (GPU/CPU)
    X_tensor = torch.tensor(X_scaled, dtype=torch.float32).to(device)
    y_tensor = torch.tensor(y_encoded, dtype=torch.long).to(device)

    return X_tensor, y_tensor, len(encoder.classes_)

def run_mass_gdv_analysis():
    print(f"Inizio analisi GDV su {len(UCI_DATASETS)} dataset...\n")
    
    risultati = []
    
    for nome, uci_id in UCI_DATASETS.items():
        print(f"Elaborazione: {nome} (ID: {uci_id})...", end=" ")
        start_time = time.time()
        
        try:
            dataset = fetch_ucirepo(id=uci_id)
            X_raw = dataset.data.features
            y_raw = dataset.data.targets
            
            campioni_totali_reali = X_raw.shape[0]
            num_feature_originali = X_raw.shape[1]

            if y_raw is None or y_raw.empty:
                raise ValueError("Nessun target (y) trovato nel dataset.")
                
            X_tensor, y_tensor, num_classes = clean_and_prepare_data(X_raw, y_raw)
            
            gdv = calculate_gdv(X_tensor, y_tensor)
            
            tempo = time.time() - start_time
            print(f"Completato! GDV: {gdv:.4f} | Tempo: {tempo:.2f}s")
            
            risultati.append({
                "Dataset": nome,
                "UCI_ID": uci_id,
                "Num_Campioni": campioni_totali_reali,
                "Num_Feature_Pre_Clean": num_feature_originali,
                "Num_Feature_Post_Clean": X_tensor.shape[1],
                "Num_Classi": num_classes,
                "GDV_Spazio_Input": round(gdv, 4),
                "Status": "OK",
                "Note_Errore": ""
            })
            
        except Exception as e:
            messaggio_errore = f"{type(e).__name__}: {str(e)}"
            print(f"FALLITO! Motivo: {messaggio_errore}")
            
            risultati.append({
                "Dataset": nome,
                "UCI_ID": uci_id,
                "Num_Campioni": None,
                "Num_Feature_Post_Clean": None,
                "Num_Classi": None,
                "GDV_Spazio_Input": None,
                "Status": "Fallito",
                "Note_Errore": messaggio_errore
            })
            
    df_report = pd.DataFrame(risultati)
    csv_filename = "report_gdv_uci_dettagliato.csv"
    df_report.to_csv(csv_filename, index=False)
    
    print("\n" + "="*60)
    print("ANALISI COMPLETATA!")
    print(f"I risultati sono stati salvati nel file: '{csv_filename}'")
    print("="*60)
    
    return df_report


if __name__ == "__main__":
    df_finale = run_mass_gdv_analysis()
    
    print("\n--- RIEPILOGO ESECUZIONE ---")
    totali = len(df_finale)
    successi = len(df_finale[df_finale['Status'] == 'OK'])
    fallimenti = totali - successi
    print(f"Dataset analizzati con successo: {successi}/{totali}")
    print(f"Dataset falliti: {fallimenti}/{totali}")
    
    if fallimenti > 0:
        print("\n--- DETTAGLIO FALLIMENTI ---")
        df_falliti = df_finale[df_finale['Status'] == 'Fallito']
        print(df_falliti[['Dataset', 'Note_Errore']].to_string(index=False))